# Module 4

## Prerequisites

In [1]:
import boto3
import json
import time
from datetime import datetime
import os

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

## Inference profile

In [4]:
prompt = """
What design patterns are used in this code?

You are an expert Python developer. Here's a codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

### Invoking cross-region inference profile

In [5]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "c4400486-751b-41a6-a030-12aba4d7b754",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 14 Sep 2026 19:52:23 GMT",
      "content-type": "application/json",
      "content-length": "4881",
      "connection": "keep-alive",
      "x-amzn-requestid": "c4400486-751b-41a6-a030-12aba4d7b754"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet, several design patterns are potentially used or could be applied to enhance the design. Here's an analysis of possible patterns within the given context:\n\n### 1. **Builder Pattern**\nThe `DataProcessor` class constructor requires a `config` object, which suggests that the class might benefit from the Builder pattern. The Builder pattern is useful when constructing complex objects, like the configuration for a `DataProcessor`, in a step-by-step manner.\n\nExample:\n`

### Use your own inference profile

In [6]:
# create an inference profile
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

response = bedrock_control.create_inference_profile(
    inferenceProfileName='dk-nova-lite-profile',
    description='Custom inference profile for Nova Lite',
    modelSource={
        'copyFrom': 'arn:aws:bedrock:us-east-1:767397992380:inference-profile/us.amazon.nova-lite-v1:0'
    }
)
inference_profile_arn = response['inferenceProfileArn']

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "f8f7dd5a-00e9-4ef9-903f-fc4c81e7e7ab",
    "HTTPStatusCode": 201,
    "HTTPHeaders": {
      "date": "Mon, 14 Sep 2026 19:53:09 GMT",
      "content-type": "application/json",
      "content-length": "125",
      "connection": "keep-alive",
      "x-amzn-requestid": "f8f7dd5a-00e9-4ef9-903f-fc4c81e7e7ab"
    },
    "RetryAttempts": 0
  },
  "inferenceProfileArn": "arn:aws:bedrock:us-east-1:767397992380:application-inference-profile/f71acc74oofn",
  "status": "ACTIVE"
}


In [7]:
response = bedrock.converse(
    modelId=inference_profile_arn,
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "e7a5cacd-3cd7-433b-a69b-41f8ccd04f3b",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 14 Sep 2026 19:54:29 GMT",
      "content-type": "application/json",
      "content-length": "5750",
      "connection": "keep-alive",
      "x-amzn-requestid": "e7a5cacd-3cd7-433b-a69b-41f8ccd04f3b"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided context, the `DataProcessor` class does not explicitly implement any well-known design patterns within the code snippet itself. However, the overall design of the class hints at several design patterns that could be applied to the larger system to improve its maintainability, scalability, and flexibility. Here are some design patterns that could be relevant:\n\n### 1. **Factory Pattern**\nGiven that there are 50+ similar classes, a Factory Pattern could be used to manage the creati

In [8]:
response = bedrock_control.delete_inference_profile(
    inferenceProfileIdentifier=inference_profile_arn
)

## Batch Inference

Upload jsonl file to S3

In [13]:
bucket_name = "batch-data-inference-dk0"

s3 = boto3.client('s3')

# Upload manifest to S3
input_key = 'input/city-reviews-manifest.jsonl'
s3.upload_file('input/city-reviews-manifest.jsonl', bucket_name, input_key)

Call the CreateModelInvocationJob API

In [14]:
accountnumber = "767397992380"

# Create the bedrock control plane client
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

# Create batch inference job
job_name = f"batch-job-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

response = bedrock_control.create_model_invocation_job(
    jobName=job_name,
    roleArn=f"arn:aws:iam::{accountnumber}:role/BedrockBatchRole",
    modelId="us.amazon.nova-lite-v1:0",
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': f's3://{bucket_name}/input/'
        }
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': f's3://{bucket_name}/output/'
        }
    }
)

job_arn = response['jobArn']


Get the Job Status

In [15]:
status_response = bedrock_control.get_model_invocation_job(jobIdentifier=job_arn)
status = status_response['status']
print(f"Job status: {status}")

Job status: Submitted


Instead of waiting, let's use a jobArn for a job we know already finished

In [16]:
completed_job_arn = job_arn

Wait for the job to finish

In [17]:
# Wait for job completion
while True:
    status_response = bedrock_control.get_model_invocation_job(jobIdentifier=completed_job_arn)
    status = status_response['status']
    print(f"Job status: {status}")
    
    if status in ['Completed', 'PartiallyCompleted', 'Failed', 'Stopped', 'Expired']:
        break
    
    time.sleep(30)



Job status: Submitted
Job status: Submitted
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Scheduled
Job status: Scheduled
Job status: Scheduled
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: Completed


Download the result

In [18]:
if status in ["Completed", "PartiallyCompleted"]:
    # Download results
    output_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix='output/')
    
    os.makedirs('batch-output', exist_ok=True)
    
    for obj in output_objects.get('Contents', []):
        if obj['Key'].endswith('/'):
            continue
            
        local_file = os.path.join('batch-output', obj['Key'].split('/')[-1])
        s3.download_file(bucket_name, obj['Key'], local_file)
        print(f"Downloaded: {local_file}")


Downloaded: batch-output\city-reviews-manifest.jsonl.out
Downloaded: batch-output\manifest.json.out


Stop the job we created to save on cost

In [ ]:
response = bedrock_control.stop_model_invocation_job(
    jobIdentifier=job_arn
)

## Prompt Caching with Nova Lite

In [ ]:
# Large context that we want to cache
cached_context = """
You are a expert Python developer. Here's a large codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

First request that establishes the cache

In [ ]:
firstRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "What design patterns are used in this code?"
                }
          ]
      }
    ]
)

print(json.dumps(firstRequest, indent=2))

Second request that uses cached context

In [ ]:
secondRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "what else can you tell me about this code?"
                }
          ]
      }
    ]
)

print(json.dumps(secondRequest, indent=2))

Compare the usage

In [ ]:
print("### First request ###")
print("Usage:", json.dumps(firstRequest['usage'], indent=2))
print("Metrics:", json.dumps(firstRequest['metrics'], indent=2))

print("\n\n### Second request ###")
print("Usage:", json.dumps(secondRequest['usage'], indent=2))
print("Metrics:", json.dumps(secondRequest['metrics'], indent=2))

## Converse on documents
### Using `bytes` as a source

In [ ]:
with open('input/AnyCompany_financial_10K.pdf', "rb") as file:
    doc_bytes = file.read()

In [ ]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "AnyCompany_financial",
                        "source": {
                            "bytes": doc_bytes
                        }
                    }
                },
                {
                    "text": "What investments have AnyCompany made?"
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

### Using s3Location as a source

In [ ]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

input_key = 'AnyCompany_financial_10K.pdf'
s3.upload_file('input/AnyCompany_financial_10K.pdf', bucket_name, input_key)

## Conversation history

### Create the DynamoDB Table and set the time to live

In [19]:
table_name = "conversation-history"

dynamodb = boto3.client('dynamodb', region_name="us-east-1")
response = dynamodb.create_table(
    TableName=table_name,
    BillingMode='PAY_PER_REQUEST',
    AttributeDefinitions=[
        {
            'AttributeName': 'userId',
            'AttributeType': 'S'
        },
        {
            'AttributeName': 'timestamp',
            'AttributeType': 'N'
        }
    ],
    KeySchema=[
        {
            'AttributeName': 'userId',
            'KeyType': 'HASH'
        },
        {
            'AttributeName': 'timestamp',
            'KeyType': 'RANGE'
        }
    ]
)

# Wait for table to be active
waiter = dynamodb.get_waiter('table_exists')
waiter.wait(TableName=table_name)

ttl_response = dynamodb.update_time_to_live(
    TableName=table_name,
    TimeToLiveSpecification={
        'AttributeName': 'ttl',
        'Enabled': True
    }
)

### Create the functions to get the conversation history and store new messages

In [20]:
table_name = "conversation-history"
ddbclient = boto3.client('dynamodb', region_name = "us-east-1")

def get_conversation_history(user_id):
    # Pagination is required in case the conversation is more than 1MB.
    paginator = ddbclient.get_paginator('query')
    pages = paginator.paginate(
        TableName = table_name,
        KeyConditionExpression = 'userId = :val',
        ExpressionAttributeValues = {':val': {'S': user_id}}
    )
    messages = []
    for page in pages:
        for item in page.get('Items', []):
            messages.append({
                "role": item["role"]["S"],
                "content": [{ "text": item["message"]["S"] }]
            })
    return messages

In [21]:
dynamodbResource = boto3.resource('dynamodb', region_name = "us-east-1")
conversation_table = dynamodbResource.Table(table_name)

def store_message(user_id, message, role):
    now_in_seconds = int(time.time())
    expire_ttl = now_in_seconds + (1 * 24 * 60 * 60) # 1 day
    
    conversation_table.put_item(
        Item = {
            'userId': user_id,
            'timestamp': now_in_seconds,
            'message': message,
            'role': role,
            'ttl': expire_ttl
        }
    )


### Create the function to send messages to the LLM

In [22]:
def send_message(user_id, message):
    
    # Load the conversation history
    conversation_messages = get_conversation_history(user_id)

    # Store the new message
    store_message(user_id, message, "user")
    
    # Add the new message to the conversation
    conversation_messages.append({
        "role": "user",
        "content": [{ "text": message }]
    })

    # Send the conversation to the LLM
    model_response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=conversation_messages,
        system = [{ 
            "text": "Please provide a helpful, conversational response based on the available information and conversation history." 
        }],
        inferenceConfig = {
            "maxTokens": 300,
            "temperature": 0.7,
            "topP": 0.9
        }
    )

    # Parse the message from the LLM, store it and return it
    assistant_msg = model_response['output']['message']['content'][0]['text']
    store_message(user_id, assistant_msg, "assistant")

    return conversation_messages, assistant_msg

### Sending messages

In [23]:
user_id = "Disha"
convo, response = send_message(user_id, "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?")
print(response)


Hi Disha! That's great to hear you love pizza—Vancouver has a fantastic pizza scene with many places to choose from. Here are some popular pizza spots in Vancouver that you might enjoy:

1. **Ravioli Rocket**  
   Located in the East End, Ravioli Rocket is famous for its thin-crust pizzas and creative toppings. They often use seasonal and locally sourced ingredients.

2. **Libretto Pizzeria**  
   Situated in Kitsilano, Libretto is known for its Neapolitan-style pizzas, made in a wood-fired oven. They offer a variety of toppings and even have vegan options.

3. **Pizzeria Numero Uno**  
   Another great spot for Neapolitan pizza, this one is located in the West End. They have a cozy atmosphere and are known for their classic margherita and prosciutto crudo.

4. **Cinepizza**  
   Cinepizza is in the Mount Pleasant area and offers a more casual dining experience with a great selection of pizzas, along with some fun décor that includes old movie posters.

5. **Forno Wood Oven Pizzeria** 

In [24]:
convo, response = send_message(user_id, "What do I love to eat?")
print(response)

It sounds like you have a strong affection for pizza! If you're in the mood for something other than pizza, Vancouver also offers a variety of other delicious foods. Here are a few suggestions based on popular cuisines in the city:

1. **Sushi**: 
   - **Sushi Ko**: Located in Chinatown, known for its fresh sushi and sashimi.
   - **Sushi Maru**: In Gastown, famous for its conveyor belt sushi.

2. **Burgers**:
   - **Burger Burger**: Located in Kitsilano, this place is known for its gourmet burgers.
   - **The Keg Steakhouse + Bar**: Offers a variety of burger options with a nice ambiance.

3. **Tacos**:
   - **Tacoroca**: A food truck that serves delicious Mexican street tacos.
   - **El Pastor**: Located in Kitsilano, known for its authentic Mexican tacos.

4. **Seafood**:
   - **The Boathouse Seafood + Bar**: Located on Granville Island, offering fresh seafood with a waterfront view.
   - **Fish & Game**: A fine dining restaurant in downtown Vancouver specializing in locally sourced

In [26]:
print(json.dumps(convo, indent=2))

[
  {
    "role": "user",
    "content": [
      {
        "text": "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "text": "Hi Disha! That's great to hear you love pizza\u2014Vancouver has a fantastic pizza scene with many places to choose from. Here are some popular pizza spots in Vancouver that you might enjoy:\n\n1. **Ravioli Rocket**  \n   Located in the East End, Ravioli Rocket is famous for its thin-crust pizzas and creative toppings. They often use seasonal and locally sourced ingredients.\n\n2. **Libretto Pizzeria**  \n   Situated in Kitsilano, Libretto is known for its Neapolitan-style pizzas, made in a wood-fired oven. They offer a variety of toppings and even have vegan options.\n\n3. **Pizzeria Numero Uno**  \n   Another great spot for Neapolitan pizza, this one is located in the West End. They have a cozy atmosphere and are known for their clas